# Design DNA constructs for the PETase recombination library

This notebook follows the original Twist construct notebook, retaining the ten natural parents. It assembles the same Golden Gate fragments, vector adaptors, and pTwist backbone and exports the synthesis sequences.

**Inputs:** amino-acid blocks, saved Twist codon-optimized sequences, and Golden Gate overhangs in `data/constructs/`.
**Outputs:** three FASTA files in `constructs/`, including `Twist_designs_full_constructs.fasta`.

Parent rows 0–9 correspond to synthesis labels `natural1`–`natural10`. The fixed random seed preserves the original spacer choices. Run all cells in order.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "data").is_dir(), "Run this notebook from the repository root."
import csv
import numpy as np
from random import choice, seed
from Bio.Seq import Seq

seed(1)
INPUT = ROOT / "data/constructs"
OUTPUT = ROOT / "constructs"
OUTPUT.mkdir(exist_ok=True)

def split_codons(sequence):
    assert len(sequence) % 3 == 0
    return [sequence[i:i+3] for i in range(0, len(sequence), 3)]

def translate(sequence):
    return str(Seq(sequence).translate())


## 1. Load the ten parent proteins

`parent_amino_acid_blocks.csv` contains the ten fixed-breakpoint rows used by the source notebook (original zero-based rows 18–27), with one column per block. Joining the blocks gives the protein sequence supplied for codon optimization.

In [2]:
with (INPUT / "parent_amino_acid_blocks.csv").open() as f:
    block_rows = list(csv.reader(f))[1:]
assert len(block_rows) == 10 and all(len(row) == 10 for row in block_rows)
blocks = [','.join(row) for row in block_rows]
AA = [''.join(row) for row in block_rows]
(OUTPUT / "parent_amino_acid_sequences.fasta").write_text(
    ''.join(f">s{i}\n{seq}\n" for i, seq in enumerate(AA)))
print(f"Loaded {len(AA)} parents, each divided into 10 blocks.")

Loaded 10 parents, each divided into 10 blocks.


## 2. Use the saved Twist codon optimization

The original workflow submitted the full ORFs to Twist for yeast codon optimization, avoiding BbsI, BsaI, and BsmBI. The saved vendor results are included so rerunning this notebook does not require the vendor service. Only rows `s0`–`s9` are needed.

Translation must exactly match the input proteins before adding assembly sequences.

In [3]:
with (INPUT / "twist_codon_optimized_parents.csv").open() as f:
    twist = list(csv.DictReader(f))
assert [row['Name'] for row in twist] == [f's{i}' for i in range(10)]
DNA = [row['Insert sequence'] for row in twist]
codons = [split_codons(seq) for seq in DNA]
assert [translate(seq) for seq in DNA] == AA
print("All 10 codon-optimized sequences translate to the input proteins.")

All 10 codon-optimized sequences translate to the input proteins.


## 3. Define restriction sites and Golden Gate junctions

Nine junction overhangs connect the ten blocks. `TAGC` and `GGAT` connect the first and last blocks to the plasmid. The first five blocks use BbsI and the last five use BsaI.

In [4]:
BbsI_5 = 'GAAGAC' # to be appended on 5' end, cuts to right
BbsI_3 = 'GTCTTC' # to be appended on 3' end, cuts to left
BsmBI_5 ='CGTCTC' # to be appended on 5' end, cuts to right
BsmBI_3 ='GAGACG' # to be appended on 3' end, cuts to left
BsaI_5 = 'GGTCTC' # to be appended on 5' end, cuts to right
BsaI_3 = 'GAGACC' # to be appended on 3' end, cuts to left

def randNT(n):
    return ''.join([choice('ACGT') for i in range(n)])

In [5]:
# load Chase's optimized GG overhangs 
GGfile = open(INPUT / 'golden_gate_overhangs.csv').read().split('\n')[1:10]
GG_overhangs = [l.split(',')[2] for l in GGfile] # the 4 bp overhang
GG_codons = [l.split(',')[3] for l in GGfile] # the full two codons containing the overhang

# create GG ends for each fragment
# GG3 to be added on the 3' end and GG5 to be added at the 5' end
# GG3: NT up to and including overhang
GG3 = [GG_codons[i][ :GG_codons[i].find(GG_overhangs[i])+4 ] for i in range(len(GG_codons))] 
# GG5: overhang and trailing NT
GG5 = [GG_codons[i][GG_codons[i].find(GG_overhangs[i]):]  for i in range(len(GG_codons))] 

vecOH1 = 'TAGC' # overhang to append at the beginning to ligate into plasmid 
vecOH2 = 'GGAT' # overhang to append at the end to ligate into plasmid

## 4. Assemble the fragments

For each parent, split the codon-optimized sequence at the protein block boundaries, substitute the Golden Gate junction codons, and add restriction sites and seeded random spacers. Padding preserves the reading frame used to inspect the constructs.

In [6]:
# Add the GG3 and GG5 overhangs to the ends of each block
NT_blocks = []
for i,seq in enumerate(blocks): # this enumerates over the parents 
    
    # determine the breakpoints along the sequence 
    breakpoints = [0]+list(np.cumsum([len(s) for s in seq.split(',')]))
    
    # split the codon parent sequence into blocks 
    bl = []
    for j in range(len(breakpoints)-1): # iterate over blocks
        cdn_slice = codons[i][breakpoints[j]:breakpoints[j+1]]
        bl.append(cdn_slice)
        
    # put the GG ends on each block (this includes the overhang sequence)
    for j in range(len(bl)-1): # iterate over junctions 
        bl[j][-1] = GG3[j] # assign GG3 to end of jth block
        bl[j+1][0] = GG5[j] # assign GG5 to beginning of (j+1)th block              
        
    # put the vector overhangs on the first block and the last block 
    bl[0] = [vecOH1]+bl[0]
    bl[-1] = bl[-1]+[vecOH2]
    
    # add 5' and 3' BbsI sites to the ends
    pad = 2 # additional NTs on ends to break up back-to-back BbsI sites 
    for j in range(len(bl)): # iterate over blocks 
        
        if j<5: # first five blocks use BbsI (2 NT spacer)
            bl[j] = [randNT(pad),BbsI_5,randNT(2)] + bl[j] + [randNT(2),BbsI_3,randNT(pad)]
        
        else: #last five blocks use BsaI (1 NT spacer)
            bl[j] = [randNT(pad),BsaI_5,randNT(1)] + bl[j] + [randNT(1),BsaI_3,randNT(pad)]
        
        # a lot of work here to keep everything in frame
        # this makes tweaking the sequence design in SnapGene easier
        # add addtional random NTs at the beginning/end to keep divisible by 3
        upstream_len = len(''.join(bl[j][:4])) # pad+REsite+spacer+GG5
        downstream_len = len(''.join(bl[j][-4:])) # GG3+spacer+REsite+pad
        
        upstream_add = 3-upstream_len%3
        downstream_add = 3-downstream_len%3
        
        if upstream_add==3: upstream_add=0
        if downstream_add==3: downstream_add=0
        
        bl[j] = [randNT(upstream_add)] + bl[j] + [randNT(downstream_add)]
    
    # join the codons to get the full block sequences 
    bl = [''.join(b) for b in bl]
    NT_blocks.append(bl)


## 5. Add vector adaptors and the pTwist backbone

The source notebook assigned the adaptors twice; the final, effective assignments are retained here. The full construct FASTA appends the same pTwist Kan high-copy backbone used in that notebook. The final adaptor assignments make the full insert length nondivisible by three; these assembly constructs are preserved exactly rather than treated as a single translated ORF.

In [7]:
pTwist_Kan_HC = "aggctaggtggaggctcagtgatgataagtctgcgatggtggatgcatgtgtcatggtcatagctgtttcctgtgtgaaattgttatccgctcagagggcacaatcctattccgcgctatccgacaatctccaagacattaggtggagttcagttcggcgtatggcatatgtcgctggaaagaacatgtgagcaaaaggccagcaaaaggccaggaaccgtaaaaaggccgcgttgctggcgtttttccataggctccgcccccctgacgagcatcacaaaaatcgacgctcaagtcagaggtggcgaaacccgacaggactataaagataccaggcgtttccccctggaagctccctcgtgcgctctcctgttccgaccctgccgcttaccggatacctgtccgcctttctcccttcgggaagcgtggcgctttctcatagctcacgctgtaggtatctcagttcggtgtaggtcgttcgctccaagctgggctgtgtgcacgaaccccccgttcagcccgaccgctgcgccttatccggtaactatcgtcttgagtccaacccggtaagacacgacttatcgccactggcagcagccactggtaacaggattagcagagcgaggtatgtaggcggtgctacagagttcttgaagtggtggcctaactacggctacactagaagaacagtatttggtatctgcgctctgctgaagccagttaccttcggaaaaagagttggtagctcttgatccggcaaacaaaccaccgctggtagcggtggtttttttgtttgcaagcagcagattacgcgcagaaaaaaaggatctcaagaagatcctttgatcttttctacggggtctgacgctctattcaacaaagccgccgtcccgtcaagtcagcgtaaatgggtagggggcttcaaatcgtccgctctgccagtgttacaaccaattaacaaattctgattagaaaaactcatcgagcatcaaatgaaactgcaatttattcatatcaggattatcaataccatatttttgaaaaagccgtttctgtaatgaaggagaaaactcaccgaggcagttccataggatggcaagatcctggtatcggtctgcgattccgactcgtccaacatcaatacaacctattaatttcccctcgtcaaaaataaggttatcaagtgagaaatcaccatgagtgacgactgaatccggtgagaatggcaaaagcttatgcatttctttccagacttgttcaacaggccagccattacgctcgtcatcaaaatcactcgcatcaaccaaaccgttattcattcgtgattgcgcctgagcgagacgaaatacgcgatcgctgttaaaaggacaattacaaacaggaatcgaatgcaaccggcgcaggaacactgccagcgcatcaacaatattttcacctgaatcaggatattcttctaatacctggaatgctgttttcccggggatcgcagtggtgagtaaccatgcatcatcaggagtacggataaaatgcttgatggtcggaagaggcataaattccgtcagccagtttagtctgaccatctcatctgtaacatcattggcaacgctacctttgccatgtttcagaaacaactctggcgcatcgggcttcccatacaatcgatagattgtcgcacctgattgcccgacattatcgcgagcccatttatacccatataaatcagcatccatgttggaatttaatcgcggcctcgagcaagacgtttcccgttgaatatggctcataacaccccttgtattactgtttatgtaagcagacagttttattgttcatgatgatatatttttatcttgtgcaatgtaacatcagagattttgagacacaacgtggctttcccccgccgctctagaactagtggatccaaataaaacgaaaggctcagtcgaaagactgggcctttcgttttatctgttgtttgtcgcattatacgagacgtccaggttgggatacctgaaacaaaacccatcgtacggccaaggaagtctccaataactgtgatccaccacaagcgccagggttttcccagtcacgacgttgtaaaacgacggccagtcatgcataatccgcacgcatctggaataaggaagtgccattccgcctgacct"

# Final adaptor sequences used in the source notebook.
upstream = 'AC CGTCTC A CTAGC GA GTCTTC CGTCTC A CTAT G GAGACC'
downstream = 'GGTCTC T GGAT CCA CGCG A GAGACG GAAGAC AT CTAT C GAGACG GC'
inserts = [(upstream + ''.join(parent) + downstream).replace(' ', '')
           for parent in NT_blocks]
print("Insert lengths modulo 3:", sorted({len(seq) % 3 for seq in inserts}))

Insert lengths modulo 3: [2]


## 6. Check restriction sites and remove overlapping Dcm motifs

These checks retain the source notebook's restriction-site counts. The original substitutions remove Dcm motifs overlapping BsaI recognition sites.

In [8]:
for enzyme, forward, reverse in [('BbsI', BbsI_5, BbsI_3),
                                  ('BsmBI', BsmBI_5, BsmBI_3),
                                  ('BsaI', BsaI_5, BsaI_3)]:
    print(enzyme, sorted({seq.count(forward) + seq.count(reverse) for seq in inserts}))

BbsI [12]
BsmBI [4]
BsaI [12]


In [9]:
# BsaI sites will frequently have Dcm methylation sites next to them
# BsaI ends with CC
# Dcm sites are CCTGG and CCAGG
# This dict has all combinations of BsaI-Dcm sites and replaces the central A/T with a G/C to remove Dcm
#       BsaI-Dcm    Replacement
Dcm = {'GAGACCTGG':'GAGACCGGG', # BsaI-Dcm
       'GAGACCAGG':'GAGACCGGG', # BsaI-Dcm
       'CCAGGTCTC':'CCCGGTCTC', # Dcm-BsaI
       'CCTGGTCTC':'CCGGGTCTC'} # Dcm-BsaI

fixed = []
for seq in inserts:
    for site in Dcm:
        seq = seq.replace(site,Dcm[site])
    fixed.append(seq)
    
inserts = fixed

## 7. Export the synthesis constructs

Each file has exactly ten records, `natural1` through `natural10`. The full constructs contain the synthesis inserts followed by the vector backbone; the inserts-only file omits that backbone.

In [10]:
assert len(inserts) == 10
for filename, sequences in [
    ('Twist_designs_inserts_only.fasta', inserts),
    ('Twist_designs_full_constructs.fasta', [s + pTwist_Kan_HC for s in inserts]),
]:
    fasta = ''.join(f">natural{i+1}\n{seq}\n" for i, seq in enumerate(sequences))
    (OUTPUT / filename).write_text(fasta)
    print(f"{filename}: {len(sequences)} records")

Twist_designs_inserts_only.fasta: 10 records
Twist_designs_full_constructs.fasta: 10 records
